# PCNN -- Beispiel 2: Open-Meteo-Wetterdaten (nur Koordinaten)

Santiago Rojo Osorio

Dieses Notebook nutzt **echte Gebäudedaten** (`Data_processed.csv`) zusammen mit
Wetterdaten von **Open-Meteo** (open-meteo.com), einer freien Wetter-API auf
Basis von Reanalyse-/Modelldaten.

Im Unterschied zu `example_01_meteoswiss.ipynb` gibt es hier **keine Stationswahl**:
nur Koordinaten (Breite/Länge) angeben, fertig. Funktioniert weltweit, nicht nur
in der Schweiz -- dafür sind es Modelldaten für den Gitterpunkt der Koordinate,
keine direkte Stationsmessung.

**Hinweis:** Dieses Notebook lädt Daten live aus dem Internet
(`archive-api.open-meteo.com`) -- es muss lokal mit Internetzugang ausgeführt werden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pcnn_model import PCNNModel, print_pcnn_result, print_energy_balance
from weather_sources import fetch_openmeteo_hourly

## 1. Konfiguration

In [ ]:
# ─── Gebäudedaten ────────────────────────────────────────────────────────────
DATA_PATH = "data/heizdaten_beispiel.csv"
QCOL      = "Waermeleistung_gemessen"        # oder "Waermeleistung"

# ─── Standort (nur Koordinaten -- keine Stationswahl nötig) ─────────────────
LAT, LON = 47.4239, 9.3767   # Beispiel: St. Gallen

# ─── Zeitraum ────────────────────────────────────────────────────────────────
# An den tatsächlich in Data_processed.csv vorhandenen Zeitraum anpassen
# (siehe Ausgabe von Zelle weiter unten: df_real.index.min() / .max()).
START = "2026-03-01"
END   = "2026-03-31"

## 2. Gebäudedaten laden (`Data_processed.csv`)

In [ ]:
df_real = pd.read_csv(
    DATA_PATH,
    usecols=["timestamp", "Waermeleistung", "Waermeleistung_gemessen"],
    parse_dates=["timestamp"],
    index_col="timestamp",
)
print(f"Datenbereich verfügbar: {df_real.index.min()}  bis  {df_real.index.max()}")

df_real_1h = df_real.resample("1h").mean()
df_real_1h = df_real_1h.loc[START:END]
print(f"Verwendeter Zeitraum : {df_real_1h.index.min()}  bis  {df_real_1h.index.max()}  "
      f"({len(df_real_1h)} Stunden)")

## 3. Wetterdaten von Open-Meteo laden

In [ ]:
weather = fetch_openmeteo_hourly(LAT, LON, START, END)
weather.head()

## 4. Gebäude- und Wetterdaten zusammenführen

In [ ]:
df_join = df_real_1h.join(weather, how="inner").dropna()
print(f"Gemeinsamer Zeitraum: {len(df_join)} Stunden")
df_join[[QCOL, "T_a", "W_s", "I_g"]].head()

## 5. PCNN trainieren

In [ ]:
model = PCNNModel(epochs=1000, verbose=True)
result = model.fit(
    df_join[QCOL].values,
    df_join["T_a"].values,
    df_join["W_s"].values,
    df_join["I_g"].values,
)
print_pcnn_result(result)

## 6. Vorhersage vs. Messung (Validierungsperiode)

In [ ]:
n_val = result.n_val
t_val = np.arange(n_val)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_val, result.Q_true, label=f"{QCOL} (real)", color="#185FA5")
ax.plot(t_val, result.Q_pred, label="PCNN Q_hat", color="#E24B4A", ls="--")
ax.set_xlabel("Stunde (Validierungsperiode)"); ax.set_ylabel("Q_h [W]")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## 7. Energieverteilung

In [ ]:
T_a_val = df_join["T_a"].values[-n_val:]
W_s_val = df_join["W_s"].values[-n_val:]
I_g_val = df_join["I_g"].values[-n_val:]

energie = model.energy_balance(T_a_val, W_s_val, I_g_val)
print_energy_balance(energie, label=f"Open-Meteo ({LAT}, {LON}) -- {START} bis {END}")